In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

#from data_scripts.data_import import BaseballVideos  # <-- your file

In [14]:
#%cd drive/MyDrive/training_videos/

In [15]:
# For reading data
import os
import numpy as np
from xml.dom import minidom

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

# For visualizing
from torchvision.io import read_image
from torchvision import tv_tensors
from torchvision.transforms.v2 import functional as F

# For model building
import torch
import torch.nn as nn

# for videos
import cv2 as cv

class BaseballVideos(torch.utils.data.Dataset):
    def __init__(self, root=None, transforms=None):
        self.root = root
        self.transforms = transforms
        # load all image files, sorting them to
        # ensure that they are aligned
        if root==None:
            self.vids = list(sorted([i for i in os.listdir(os.path.curdir) if '.mov' in i]))
            self.notes = list(sorted([i for i in os.listdir(os.path.curdir) if '.xml' in i]))
            if len(self.vids)!=len(self.notes):
                raise RuntimeError("Mismatch of annotation files and video files.\nPlease confirm that you have one annotation file for each video and try again.")
        imgs = []
        notes = []
        for i, k in zip(self.vids, self.notes):
            cap = cv.VideoCapture(i)
            note = minidom.parse(k)
            ret = True
            frame_count = 0
            while ret:
              ret, frame = cap.read()
              if ret:
                frame_count += 1
                frame = np.moveaxis(frame, -1, 0) # Pivot image so color channels first, then H then W
                imgs.append(torch.from_numpy(frame))
                canvas_size = list(frame.shape[1:])

            for f in range(frame_count):
                frame_i = [j for j in note.getElementsByTagName("box") if int(j.attributes['frame'].value)==f]
                boxes = []
                labels = []
                areas = []
                movings = []

                for j in frame_i:
                    moving = j.getElementsByTagName('attribute')[0].firstChild.data=='true'

                    xtl = float(j.attributes['xtl'].value)
                    ytl = float(j.attributes['ytl'].value)
                    xbr = float(j.attributes['xbr'].value)
                    ybr = float(j.attributes['ybr'].value)
                    box = (xtl, ytl, xbr, ybr)

                    label = 'baseball'
                    area = (xbr - xtl) * (ybr - ytl)

                    boxes.append(box)
                    labels.append(label)
                    areas.append(area)
                    movings.append(moving)

                target = {}
                target["boxes"] = tv_tensors.BoundingBoxes(boxes, format="XYXY", canvas_size=canvas_size)
                target["labels"] = labels
                target["area"] = areas
                target["moving"] = movings

                notes.append(target)
        self.imgs = imgs
        self.notes = notes

            # target = {}
            # target["boxes"] = tv_tensors.BoundingBoxes(boxes, format="XYXY", canvas_size=F.get_size(img))
            # target["masks"] = tv_tensors.Mask(masks)
            # target["labels"] = labels
            # target["image_id"] = image_id
            # target["area"] = area
            # target["iscrowd"] = iscrowd

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):

        img = self.imgs[idx]
        target = self.notes[idx]

        if self.transforms is not None:
            img, target = self.transforms(img, target)

        return img, target

In [16]:
# -----------------------------
# 1) Basic config (keep memory low)
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 4        # small batch -> less GPU/CPU RAM
NUM_EPOCHS = 5
LR = 1e-3

In [17]:
# -----------------------------
# 2) Dataset + DataLoader
# -----------------------------
train_dataset = BaseballVideos(
    root=None,              # or set to your videos folder
    #transforms=simple_transform
)

def collate_fn(batch):
    """
    batch = list of (img, target)
    """
    imgs, targets = zip(*batch)

    # Stack and convert to float in [0,1]
    imgs = torch.stack(imgs, dim=0).to(torch.float32) / 255.0

    labels = []
    for t in targets:
        # t["moving"] is a list of booleans, one per box
        if len(t["moving"]) == 0:
            labels.append(0.0)
        else:
            labels.append(1.0 if any(t["moving"]) else 0.0)
    labels = torch.tensor(labels, dtype=torch.float32)

    return imgs, labels


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,       # 0 keeps things simple & predictable
    collate_fn=collate_fn
)

In [18]:
# -----------------------------
# 3) Tiny CNN for "moving vs not moving"
# -----------------------------

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),  # [B, 64, 1, 1]
        )
        self.fc = nn.Linear(64, 1)    # output: logit for "moving"

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)     # [B, 64]
        x = self.fc(x)                # [B, 1]
        return x.squeeze(1)           # [B]

model = SmallCNN().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

In [27]:


# # Optionally resize frames down to keep memory low
# def simple_transform(img, target):
#     # img: [C,H,W], uint8
#     img = img.float() / 255.0          # scale to [0,1]
#     # You can resize smaller if needed:
#     # from torchvision.transforms.v2 import functional as F
#     # img = F.resize(img, [224, 224])
#     return img, target





# -----------------------------
# 4) Training loop
# -----------------------------
def train():
    model.train()
    for epoch in range(NUM_EPOCHS):
        running_loss = 0.0
        for imgs, labels in train_loader:
            imgs = imgs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            logits = model(imgs)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * imgs.size(0)

        epoch_loss = running_loss / len(train_dataset)
        print(f"Epoch {epoch+1}/{NUM_EPOCHS} - loss: {epoch_loss:.4f}")

    # -------------------------
    # 5) Save model + optimizer
    # -------------------------
    checkpoint = {
        "epoch": NUM_EPOCHS,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    }
    torch.save(checkpoint, "moving_classifier4.pth")
    print("Saved checkpoint to moving_classifier4.pth")


In [ ]:
if __name__ == "__main__":
    train()

Epoch 1/5 - loss: 0.3695
Epoch 2/5 - loss: 0.2868
Epoch 3/5 - loss: 0.2982
Epoch 4/5 - loss: 0.2868
Epoch 5/5 - loss: 0.3096
Saved checkpoint to moving_classifier.pth


In [28]:
# For reloading
import torch
#from train_moving import SmallCNN  # or redefine the same class here

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SmallCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

checkpoint = torch.load("moving_classifier3.pth", map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
start_epoch = checkpoint["epoch"]

model.eval()  # ready for inference
print(f"Restored model from epoch {start_epoch}")


Restored model from epoch 5


In [21]:
# 2nd round of training
if __name__ == "__main__":
    train()

Epoch 1/5 - loss: 0.4086
Epoch 2/5 - loss: 0.3954
Epoch 3/5 - loss: 0.3959
Epoch 4/5 - loss: 0.4012
Epoch 5/5 - loss: 0.3931
Saved checkpoint to moving_classifier2.pth


In [26]:
# 3rd round of training
if __name__ == "__main__":
    train()

Epoch 1/5 - loss: 0.3939
Epoch 2/5 - loss: 0.3957
Epoch 3/5 - loss: 0.3957
Epoch 4/5 - loss: 0.3933
Epoch 5/5 - loss: 0.3945
Saved checkpoint to moving_classifier3.pth


In [29]:
# 4th round of training
if __name__ == "__main__":
    train()

Epoch 1/5 - loss: 0.3853
Epoch 2/5 - loss: 0.3980
Epoch 3/5 - loss: 0.3904
Epoch 4/5 - loss: 0.3865
Epoch 5/5 - loss: 0.3837
Saved checkpoint to moving_classifier4.pth
